# Notebook 11 — Full independent transfer-learning retraining (revised protocol)

**Experiments:** E1 = Pakistan/Xception; E2 = Pakistan/ConvNeXt-Tiny; E3 = Stanford/Xception; E4 = Stanford/ConvNeXt-Tiny.

**No previous model checkpoints are loaded.** Each experiment starts from the named ImageNet-1K pretrained `timm` model with a fresh 48- or 196-class classifier, then trains **5 head-only epochs + up to 40 fine-tuning epochs**, with two-epoch warm-up and cosine LR decay.

**Methodological disclosure:** This revised protocol was proposed after inspecting Notebook 9's original test performance. The original test split has therefore been seen during model-development decisions. We select checkpoints using **validation only**; any revised test evaluation must be labelled *post-hoc*, not newly untouched. Preserve Notebook 9 as the audit trail even if this report emphasizes Notebook 11.

Never use final test results to change the recipe again.

## 1. Verify T4 / GPU runtime
Select Runtime → Change runtime type → T4 GPU; Connect.

In [ ]:
import sys, subprocess, torch
from pathlib import Path
assert torch.cuda.is_available(), 'Colab runtime must have GPU enabled'
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory GiB:', round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2))
subprocess.run(['nvidia-smi'], check=True)

## 2. Mount Drive — keep old checkpoints and CSVs untouched
Requires original `MyDrive/vehicle_project/data.zip`. Revised outputs go under **`revised_runs`**, not `runs`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE=Path('/content/drive/MyDrive/vehicle_project')
DATA_ZIP=DRIVE/'data.zip'
OUTPUT=DRIVE/'revised_runs'
assert DATA_ZIP.is_file(), f'Missing dataset archive: {DATA_ZIP}'
OUTPUT.mkdir(exist_ok=True, parents=True)
print('Dataset archive:',DATA_ZIP)
print('Revised run outputs:',OUTPUT)
assert OUTPUT != DRIVE/'runs'
assert (DRIVE/'runs').is_dir(), 'Keep old Notebook 9 runs backed up on Drive' 

## 3. Clone current GitHub revision
Your repository is private. If cloning fails, first authenticate GitHub CLI (`gh auth login` and `gh auth setup-git`) in this runtime; never paste access tokens into notebook text. Start with a *fresh* clone to make sure the new files are present. Keep the commit unchanged throughout runs.

In [ ]:
import os, subprocess, sys
REPO=Path('/content/vehicle-make-model-recognition')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/Batsy1857/vehicle-make-model-recognition.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'rev-parse','HEAD'],check=True)
assert (REPO/'src/train_vehicle_retrain.py').is_file(), 'Commit src/train_vehicle_retrain.py before running'
assert (REPO/'configs/retraining.yaml').is_file(), 'Commit configs/retraining.yaml before running'
subprocess.run([sys.executable,'-m','pip','install','-q','timm','pyyaml','scikit-learn','pandas','matplotlib'],check=True)
import yaml, timm, torchvision, sklearn
print('timm:',timm.__version__,'torchvision:',torchvision.__version__,'scikit-learn:',sklearn.__version__)

## 4. Restore exact prepared images into local Colab storage
No re-splitting, no re-download, no additional duplicate-filtering. We use the repository's frozen manifests.

In [ ]:
import zipfile
if not (REPO/'data/pakistan').is_dir() or not (REPO/'data/stanford_cars').is_dir():
    with zipfile.ZipFile(DATA_ZIP) as archive:
        names=archive.namelist()
        assert any(s.startswith('data/pakistan/') for s in names)
        assert any(s.startswith('data/stanford_cars/') for s in names)
        archive.extractall(REPO)
print('Dataset directory:',REPO/'data')

## 5. Verify both frozen manifests + the revised recipe

In [ ]:
import sys
sys.path.insert(0,str(REPO/'src'))
from vehicle_data import inspect_manifest
with (REPO/'configs/experiments.yaml').open() as f:
    cfg=yaml.safe_load(f)
with (REPO/'configs/retraining.yaml').open() as f:
    recipe=yaml.safe_load(f)
for ds in ('pakistan','stanford_cars'):
    info=inspect_manifest(REPO,ds,check_files=True)
    assert info['manifest_sha256']==cfg['datasets'][ds]['manifest_sha256']
    for key,n in cfg['datasets'][ds]['verified_split_counts'].items():
        observed='excluded_test_overlap' if ds=='pakistan' and key=='excluded' else key
        assert info['counts'].get(observed,0)==n
    print(ds,info['counts'],'manifest hash PASS')
assert recipe['stages']['partial_fine_tune']['epochs']['stanford_cars']==40
assert recipe['stages']['partial_fine_tune']['epochs']['pakistan']==40
print('REVISED PROTOCOL READY. No old checkpoints loaded.')

## 6. Visualize the planned learning-rate schedule
Stage 1: classifier/head only at 1e-3 for 5 epochs. Stage 2: final backbone stage + classifier, up to 40 epochs with warm-up to 3e-5 and cosine decay toward 1e-6.

In [ ]:
from train_vehicle_retrain import epoch_lr
import matplotlib.pyplot as plt
ft=recipe['stages']['partial_fine_tune']
epochs=ft['epochs']['stanford_cars']
rates=[epoch_lr(i,epochs,ft['peak_learning_rate'],ft['minimum_learning_rate'],
                ft['warmup_epochs'],ft['warmup_start_fraction']) for i in range(epochs)]
plt.figure(figsize=(8,3));plt.plot(range(1,epochs+1),rates,marker='.',linewidth=1)
plt.xlabel('Fine-tuning epoch');plt.ylabel('Learning rate');plt.title('Planned warm-up and cosine decay')
plt.grid(alpha=.25);plt.show()
print('First/peak/final LR:', rates[0],max(rates),rates[-1])

## 7. GPU smoke tests, using brand-new models
Each smoke test: 1 short head-only epoch and 1 short fine-tuning epoch; 2 train + 2 validation batches per stage. **Not performance results.**

In [ ]:
SMOKES=['E1','E4']
for exp in SMOKES:
    rid=f'{exp}-v2-smoke01'
    cmd=[sys.executable,str(REPO/'src/train_vehicle_retrain.py'),
         '--experiment',exp,'--run-id',rid,'--output-dir',str(OUTPUT),'--smoke']
    print('SMOKE:',exp,flush=True)
    subprocess.run(cmd,cwd=REPO,check=True)
print('Smoke test completed for E1 and E4')

## 8. Train one complete experiment at a time
Run **E1 → E2 → E3 → E4** (or start with Stanford E3 and E4 if the deadline requires prioritizing). Each uses its own new run ID. The full training set and full validation set are processed each epoch. The selected checkpoint maximizes validation macro-F1 across both stages.

On a disconnect: reconnect, clone the **same Git commit**, restore data, set `RESUME=True`, and run with the **same run ID**. Do not use `RESUME=True` after run completion. Never restart by deleting finished checkpoints.

In [ ]:
EXPERIMENT='E3'        # Change to E4, E1, E2 as each run completes
RUN_ID=f'{EXPERIMENT}-v2-run01'
RESUME=False           # True ONLY to continue an interrupted, incomplete run
cmd=[sys.executable,str(REPO/'src/train_vehicle_retrain.py'),
     '--experiment',EXPERIMENT,'--run-id',RUN_ID,'--output-dir',str(OUTPUT)]
if RESUME:
    cmd.append('--resume')
print('FULL REVISED TRAINING:',EXPERIMENT,RUN_ID,flush=True)
subprocess.run(cmd,cwd=REPO,check=True)
print('Saved:',OUTPUT/RUN_ID)

## 9. Inspect training, learning curves and selected checkpoint (validation only)

In [ ]:
import pandas as pd, json
info=json.loads((OUTPUT/RUN_ID/'metadata.json').read_text())
history=pd.read_csv(OUTPUT/RUN_ID/'history.csv')
display(history[['stage','epoch','learning_rate','train_loss','val_loss',
                 'val_accuracy','val_macro_f1','train_seconds','peak_gpu_memory_mb']])
print('Status:',info['status'],'best stage:',info.get('selected_stage'),
      'epoch:',info.get('selected_epoch'), 'best val macro-F1:',info.get('best_validation_macro_f1'))
assert info['status']=='completed' and (OUTPUT/RUN_ID/'best.pt').is_file()
fig, axes=plt.subplots(1,2,figsize=(10,3.5))
for stage,df in history.groupby('stage',sort=False):
    axes[0].plot(range(1,len(df)+1),df['val_macro_f1'],marker='.',label=stage)
    axes[1].plot(range(1,len(df)+1),df['train_loss'],marker='.',label=stage)
for a in axes:a.legend();a.grid(alpha=.25);a.set_xlabel('Epoch within stage')
axes[0].set_title('Validation macro-F1');axes[1].set_title('Training cross entropy')
plt.tight_layout();plt.show()

## 10. Validation-only comparison across all completed revised runs
No test data is read. This is the appropriate place to inspect convergence and choose/report the recipe.

In [ ]:
summary=[]
for exp in ('E1','E2','E3','E4'):
    path=OUTPUT/f'{exp}-v2-run01'/'metadata.json'
    if not path.is_file():continue
    i=json.loads(path.read_text())
    if i.get('status')!='completed':continue
    summary.append({'experiment':exp,'dataset':i['experiment_definition']['dataset'],
      'model':i['experiment_definition']['model'],
      'best_val_macro_f1':i['best_validation_macro_f1'],
      'best_stage':i['selected_stage'],'best_epoch':i['selected_epoch'],
      'head_epochs':i['epochs_completed_head_only'],
      'fine_epochs':i['epochs_completed_finetune'],
      'train_seconds':i['total_train_seconds_excluding_validation'],
      'gpu':i['gpu'],'git_commit':i['git_commit']})
display(pd.DataFrame(summary))

## 11. Optional: post-hoc test evaluation — only after all four checkpoint selections are frozen

**Explicit disclosure:** We already looked at Notebook 9 test results when choosing this recipe. Therefore the 751/8041-image original test sets are **not newly unseen for the revised research process**. If you evaluate revised models on these splits, label the results *post-hoc test scores* and never use them to tune/retrain again. This section is disabled by default; it refuses repeated test evaluations.

In [ ]:
RUN_POSTHOC_TESTS=False
if RUN_POSTHOC_TESTS:
    for exp in ('E1','E2','E3','E4'):
        rid=f'{exp}-v2-run01'
        info=json.loads((OUTPUT/rid/'metadata.json').read_text())
        assert info['status']=='completed', f'Incomplete {rid}'
    for exp in ('E1','E2','E3','E4'):
        rid=f'{exp}-v2-run01'
        cmd=[sys.executable,str(REPO/'src/train_vehicle_retrain.py'),
            '--experiment',exp,'--run-id',rid,'--output-dir',str(OUTPUT),
            '--mode','test','--acknowledge-posthoc-test']
        subprocess.run(cmd,cwd=REPO,check=True)
else:
    print('Test evaluation disabled. Only validation metrics are available for this revised recipe.')

## 12. Optional: matched-GPU FP32 model-forward latency
Benchmarks use random synthetic tensors, not data-loader images. All four runs should be on the same GPU model for fair timing comparisons.

In [ ]:
RUN_BENCHMARKS=False
if RUN_BENCHMARKS:
    for exp in ('E1','E2','E3','E4'):
        rid=f'{exp}-v2-run01'
        cmd=[sys.executable,str(REPO/'src/train_vehicle_retrain.py'),
             '--experiment',exp,'--run-id',rid,'--output-dir',str(OUTPUT),'--mode','benchmark']
        subprocess.run(cmd,cwd=REPO,check=True)
else:
    print('Inference benchmarking not requested.')

## 13. Export one revised results CSV for reporting
The CSV includes validation-only measurements even before post-hoc testing. Test columns remain blank until saved evaluation files exist. Save it on Drive and commit a copy to GitHub; do not commit large `.pt` checkpoints.

In [ ]:
rows=[]
for exp in ('E1','E2','E3','E4'):
    d=OUTPUT/f'{exp}-v2-run01'
    if not (d/'metadata.json').is_file():continue
    m=json.loads((d/'metadata.json').read_text())
    if m['status']!='completed':continue
    row={'experiment':exp,'dataset':m['experiment_definition']['dataset'],
         'model':m['experiment_definition']['model'],'git_commit':m['git_commit'],
         'gpu':m['gpu'],'manifest_sha256':m['manifest_sha256'],
         'total_parameters':m['parameters_head_only']['total'],
         'head_trainable_parameters':m['parameters_head_only']['trainable'],
         'fine_tune_trainable_parameters':m['parameters_partial_fine_tune']['trainable'],
         'best_val_macro_f1':m['best_validation_macro_f1'],
         'best_stage':m['selected_stage'],'best_epoch':m['selected_epoch'],
         'train_seconds_excl_validation':m['total_train_seconds_excluding_validation'],
         'peak_gpu_memory_mb':m['peak_gpu_memory_mb'],
         'test_disclosure':'Post-hoc: original test scores were previously inspected'}
    if (d/'test_metrics.json').is_file():
        t=json.loads((d/'test_metrics.json').read_text())['test_metrics']
        row.update(test_top1_accuracy=t['accuracy'],test_top5_accuracy=t['top5_accuracy'],test_macro_f1=t['macro_f1'])
    if (d/'benchmark.json').is_file():
        b=json.loads((d/'benchmark.json').read_text())
        row.update({k:v for k,v in b.items() if 'ms_per_image' in k})
    rows.append(row)
df=pd.DataFrame(rows)
export=DRIVE/'revised_metrics.csv'
df.to_csv(export,index=False)
display(df)
print('Exported:',export)

## Reporting guidance

You may dedicate the **single main performance table** in your four-page report to these four revised, matched-protocol experiments. The original Notebook 9 runs remain archived as pilots. For transparency, include a concise methods limitation such as:

> “After preliminary pilot runs, we revised the fine-tuning budget and introduced warm-up plus cosine decay. Because the original test scores were inspected before revising the protocol, revised test results are reported as post-hoc and may reflect model-development selection bias.”

Do not call the revised protocol statistically superior without repeated seeds or a fresh independent test set. The new script does not rely on Notebook 9's model checkpoints.